# EZNX-ATLAS-A — Scientific Reports: Complete Training Campaign

**170 descriptors / 150 unique training runs** covering primary ablation (3 variants × 20 seeds),  
architecture sensitivity (GLU-width), loss-weight sensitivity (LAUC),  
and data-augmentation sensitivity.

## Run order — execute cells in sequence: 0 → 1 → 2 → 3 → 4 → 5 → 6 → 7 → 8

| Cell | What it does | Must be done first |
|------|-------------|--------------------|
| 0 | GPU / env check | — |
| 1 | Install wfdb + pyarrow | — |
| 2 | Path config + env vars | Add PTB-XL dataset |
| 3 | **Bootstrap code from GitHub** | PTB-XL added + Internet ON |
| 4 | Build index_complete.parquet | Cell 2 + Cell 3 |
| 5 | Dry-run: print 170-run plan | Cell 4 |
| 6 | **Run all 170 training experiments** | Cells 0–5 |
| 7 | Post-training evaluation | All 170 runs done |
| 8 | Package results for download | Cell 7 done |

**Idempotent**: restart after session timeout and re-run Cell 6. Already-complete runs are skipped.  
**Seeds**: 2024–2043 (20 consecutive, year of study initiation; no cherry-picking).

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 0 — GPU / Environment Verification                               ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys, os

print('=== GPU ===')
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader

import torch
print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.version.cuda}')
print(f'GPU avail: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU name : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory // 1024**2} MB')

import numpy as np, sklearn, scipy
print(f'NumPy    : {np.__version__}')
print(f'sklearn  : {sklearn.__version__}')
print(f'scipy    : {scipy.__version__}')
print('=== OK ===')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 1 — Install dependencies                                         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# wfdb is the only package not pre-installed on Kaggle Python images.
# tqdm, scipy, scikit-learn, pandas, pyarrow are pre-installed.
!pip install --quiet 'wfdb==4.3.0'
!pip install --quiet 'pyarrow>=14.0'

# Verify wfdb installed correctly
import wfdb
print(f'wfdb     : {wfdb.__version__}')
print('Dependencies ready.')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 2 — Path configuration                                           ║
# ║                                                                         ║
# ║  BEFORE running this cell:                                             ║
# ║  Add PTB-XL dataset via Notebook settings → Data → Add Data           ║
# ║  Recommended: garethwmch/ptb-xl-1-0-3 or any PTB-XL 1.0.3 mirror     ║
# ╚══════════════════════════════════════════════════════════════════════════╝
from pathlib import Path
import glob, os

KAGGLE_INPUT = Path('/kaggle/input')
WORKING      = Path('/kaggle/working')

# Auto-detect ptbxl_database.csv (works with any Kaggle PTB-XL dataset slug)
matches = sorted(glob.glob(
    str(KAGGLE_INPUT / '**' / 'ptbxl_database.csv'), recursive=True
))
if not matches:
    raise FileNotFoundError(
        'ptbxl_database.csv not found under /kaggle/input.\n'
        'Please add a PTB-XL 1.0.3 dataset to this notebook first.'
    )

# Also verify scp_statements.csv is in the same directory
PTB_XL_ROOT = Path(matches[0]).parent
if not (PTB_XL_ROOT / 'scp_statements.csv').exists():
    raise FileNotFoundError(
        f'scp_statements.csv not found next to ptbxl_database.csv in {PTB_XL_ROOT}'
    )

INDEX_PATH  = WORKING / 'index_complete.parquet'
RUNS_DIR    = WORKING / 'runs'
RESULTS_DIR = WORKING / 'results'

RUNS_DIR.mkdir(exist_ok=True)
RESULTS_DIR.mkdir(exist_ok=True)

print(f'PTB-XL root : {PTB_XL_ROOT}')
print(f'Index path  : {INDEX_PATH}')
print(f'Runs dir    : {RUNS_DIR}')
print(f'Results dir : {RESULTS_DIR}')

# Confirm key PTB-XL files
for fname in ['ptbxl_database.csv', 'scp_statements.csv']:
    print(f'  ✓ {PTB_XL_ROOT / fname}')

# Confirm records directories exist
for rec_dir in ['records100', 'records500']:
    p = PTB_XL_ROOT / rec_dir
    status = '✓' if p.exists() else '✗ MISSING'
    print(f'  {status} {p}')

# Set environment variables inherited by all subprocess calls
os.environ['EZNX_DATA_REAL']  = str(PTB_XL_ROOT)
os.environ['EZNX_INDEX_PATH'] = str(INDEX_PATH)
os.environ['EZNX_RUNS_DIR']   = str(RUNS_DIR)
print('\nEnvironment variables set.')

In [ ]:
# ????????????????????????????????????????????????????????????????????????????
# ?  CELL 3 ? Bootstrap code from GitHub                                   ?
# ?                                                                        ?
# ?  BEFORE running this cell:                                             ?
# ?  Turn Internet ON in Notebook settings.                                ?
# ?  If needed, edit REPO_URL / REPO_REF / REPO_SUBDIR below.              ?
# ????????????????????????????????????????????????????????????????????????????
import os, shutil, subprocess
from pathlib import Path

REPO_URL = os.environ.get('EZNX_REPO_URL', 'https://github.com/ezynsegnane/ezyx-atlas-a_github_code.git')
REPO_REF = os.environ.get('EZNX_REPO_REF', '')  # leave empty for the repo default branch
REPO_SUBDIR = os.environ.get('EZNX_REPO_SUBDIR', 'kaggle_train')
CLONE_DIR = WORKING / 'repo_src'
required_scripts = [
    'atlas_a_v5_multiseed_v2.py',
    'run_experiments_v2.py',
    'compute_calibration.py',
    'compute_subgroups.py',
    'analyze_multiseed_v2.py',
    'evaluate_missingness_v2.py',
    'eznx_loader_v2.py',
    'eznx_model_v5.py',
    'index_construction.py',
]

clone_cmd = ['git', 'clone', '--depth', '1']
if REPO_REF:
    clone_cmd += ['--branch', REPO_REF]
clone_cmd += [REPO_URL, str(CLONE_DIR)]

if CLONE_DIR.exists():
    shutil.rmtree(CLONE_DIR)

print('Cloning training code from GitHub...')
print('  REPO_URL   =', REPO_URL)
print('  REPO_REF   =', REPO_REF or '<default branch>')
print('  REPO_SUBDIR=', REPO_SUBDIR)
subprocess.run(clone_cmd, check=True)

candidate_dirs = []
if REPO_SUBDIR:
    candidate_dirs.append(CLONE_DIR / REPO_SUBDIR)
candidate_dirs.append(CLONE_DIR)

CODE_DIR = None
for candidate in candidate_dirs:
    if all((candidate / s).exists() for s in required_scripts):
        CODE_DIR = candidate
        break

if CODE_DIR is None:
    print('\nChecked candidate directories:')
    for candidate in candidate_dirs:
        print(' -', candidate)
    raise FileNotFoundError(
        'Could not find the Kaggle training scripts in the cloned repo. '
        'Push the kaggle_train/ directory to GitHub and set REPO_SUBDIR if needed.'
    )

print('\nCODE_DIR =', CODE_DIR)
for s in required_scripts:
    size_kb = (CODE_DIR / s).stat().st_size // 1024
    print(f'  ? {s:<45} {size_kb:>5} KB')

os.environ['EZNX_CODE_DIR'] = str(CODE_DIR)
print('\nEnvironment variable set: EZNX_CODE_DIR')


In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 4 — Build index_complete.parquet                                 ║
# ║                                                                         ║
# ║  Runs index_construction.py (two-step pipeline):                       ║
# ║    Step 1: engineer metadata features → index_mm_core.parquet          ║
# ║    Step 2: merge scp_codes + filename_lr/hr → index_complete.parquet   ║
# ║                                                                         ║
# ║  Idempotent: skips if index_complete.parquet already exists.           ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

if INDEX_PATH.exists():
    print(f'index_complete.parquet already exists — skipping rebuild.')
else:
    print('Building index (Step 1: features, Step 2: merge labels) …')
    result = subprocess.run([
        sys.executable, str(CODE_DIR / 'index_construction.py'),  # dynamic path
        '--data-root', str(PTB_XL_ROOT),
        '--out-dir',   str(WORKING),
    ], capture_output=True, text=True)
    print(result.stdout[-4000:] if result.stdout else '(no stdout)')
    if result.returncode != 0:
        print('STDERR:', result.stderr[-2000:])
        raise RuntimeError('index_construction.py failed — see STDERR above.')

# Verify output
import pandas as pd
idx = pd.read_parquet(INDEX_PATH)
print(f'\nindex_complete.parquet: {len(idx)} rows, {len(idx.columns)} columns')
print('Split distribution:')
print(idx.groupby('strat_fold').size().rename('count').to_string())

# Hard integrity checks — these must all pass
required_cols = [
    'ecg_id', 'patient_id', 'strat_fold',
    'scp_codes', 'filename_lr', 'filename_hr',
    'age_z', 'sex01', 'meta_present_strict',
]
missing_cols = [c for c in required_cols if c not in idx.columns]
assert not missing_cols, f'CRITICAL: missing columns: {missing_cols}'
assert idx['strat_fold'].between(1, 10).all(), 'strat_fold out of range'
assert len(idx) == 21799, f'Expected 21799 rows, got {len(idx)}'
print(f'\nAll integrity checks passed ✓  ({len(idx)} records, {len(idx.columns)} columns)')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 5 — Dry run: print the full 170-experiment plan                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys
result = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_XL_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--dry_run',
], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('STDERR:', result.stderr[-1000:])

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 6 — Execute all 170 training runs                                ║
# ║                                                                         ║
# ║  IDEMPOTENT: already-complete runs are skipped automatically.           ║
# ║  Session timeout? Restart kernel and re-run from Cell 2 → Cell 6.     ║
# ║  (Cells 2+3 are fast; Cell 4 is skipped if parquet exists)            ║
# ║                                                                         ║
# ║  Estimated time on Kaggle P100:                                        ║
# ║    ~8–10 min/run × 150 unique runs ≈ 20–25 h (2–3 sessions)         ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

# Change --group to run a specific group only: A | B | C | D
# Leave as 'all' to run everything (recommended).
result = subprocess.run([
    sys.executable, str(CODE_DIR / 'run_experiments_v2.py'),
    '--data_root',  str(PTB_XL_ROOT),
    '--index_path', str(INDEX_PATH),
    '--runs_dir',   str(RUNS_DIR),
    '--group',      'all',
    '--resume_csv', str(RESULTS_DIR / 'progress.csv'),
], check=False)  # check=False: progress is saved per run; don't abort on single failure

print(f'\nOrchestrator exit code: {result.returncode}')
if result.returncode != 0:
    print('One or more runs may have failed. Check progress.csv for details.')
print(f'Progress CSV: {RESULTS_DIR / "progress.csv"}')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 7 — Post-training evaluation                                     ║
# ║  Run only AFTER the campaign in Cell 6 is complete.                  ║
# ╚══════════════════════════════════════════════════════════════════════════╝
import subprocess, sys

def run_eval(label, cmd):
    print(f'\n=== {label} ===')
    r = subprocess.run(cmd, capture_output=True, text=True)
    if r.stdout:
        print(r.stdout[-3000:])
    if r.returncode != 0:
        print(f'STDERR ({label}):', r.stderr[-1000:])
        print(f'WARNING: {label} exited with code {r.returncode}')
    return r.returncode

run_eval('[7a] Statistical analysis (primary + sensitivity)', [
    sys.executable, str(CODE_DIR / 'analyze_multiseed_v2.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'stats'),
])

run_eval('[7b] Calibration metrics (Brier + ECE)', [
    sys.executable, str(CODE_DIR / 'compute_calibration.py'),
    '--runs_dir', str(RUNS_DIR),
    '--out_dir',  str(RESULTS_DIR / 'calibration'),
])

run_eval('[7c] Subgroup AUC (sex / age / anthropometric)', [
    sys.executable, str(CODE_DIR / 'compute_subgroups.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--out_dir',    str(RESULTS_DIR / 'subgroups'),
])

run_eval('[7d] Missingness robustness stress-test', [
    sys.executable, str(CODE_DIR / 'evaluate_missingness_v2.py'),
    '--runs_dir',   str(RUNS_DIR),
    '--index_path', str(INDEX_PATH),
    '--data_root',  str(PTB_XL_ROOT),
    '--out_dir',    str(RESULTS_DIR / 'missingness'),
])

print('\n=== Evaluation complete. ===')
import os
for root, dirs, files in os.walk(str(RESULTS_DIR)):
    for f in sorted(files):
        rel  = os.path.relpath(os.path.join(root, f), str(RESULTS_DIR))
        size = os.path.getsize(os.path.join(root, f))
        print(f'  {rel:<60} {size:>10,} B')

In [ ]:
# ╔══════════════════════════════════════════════════════════════════════════╗
# ║  CELL 8 — Package results for download                                 ║
# ╚══════════════════════════════════════════════════════════════════════════╝
# JSON-only zip (~10 MB): sufficient for all statistical analyses.
# Set INCLUDE_NPZ = True (~3 GB) only if you need raw probability arrays.
INCLUDE_NPZ = False

import zipfile, os
from pathlib import Path

zip_path = WORKING / 'results_package.zip'
n_files  = 0

with zipfile.ZipFile(zip_path, 'w', compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(Path(RUNS_DIR).rglob('results_*.json')):
        zf.write(p, p.relative_to(WORKING)); n_files += 1
    if INCLUDE_NPZ:
        for p in sorted(Path(RUNS_DIR).rglob('probs_*.npz')):
            zf.write(p, p.relative_to(WORKING)); n_files += 1
    for p in sorted(RESULTS_DIR.rglob('*')):
        if p.is_file():
            zf.write(p, p.relative_to(WORKING)); n_files += 1

zip_mb = zip_path.stat().st_size / 1024**2
print(f'Package: {zip_path}')
print(f'  {n_files} files,  {zip_mb:.1f} MB')
print('\nDownload from Kaggle output panel → results_package.zip')

## After downloading results_package.zip

Send the zip to the study lead. The following scripts will produce all manuscript tables:

| Script | Output |
|--------|--------|
| `analyze_multiseed_v2.py` | Primary ablation table, Wilcoxon tests, sensitivity tables |
| `compute_calibration.py` | Brier score + ECE calibration table |
| `compute_subgroups.py` | Sex / age / anthropometric subgroup AUC table |
| `evaluate_missingness_v2.py` | MCAR missingness robustness table |

### Hardware provenance
- Platform: Kaggle Notebooks — GPU: NVIDIA Tesla P100-PCIE-16GB  
- Seeds: 2024–2043, CUBLAS deterministic mode (`CUBLAS_WORKSPACE_CONFIG=:4096:8`), `num_workers=0`